# Predicting Product Sales
## 6. Model Training & Benchmarking

We train four official Gradient Boosting regression architectures alongside our own custom-built **LightGBM from scratch** model:
- Scikit-learn HistGradientBoostingRegressor
- XGBoost Regressor
- LightGBM Regressor
- CatBoost Regressor
- **LightGBM from scratch** (implemented below)

For each model, we will record the training duration (fit time).

In [ ]:
import pandas as pd
import joblib
import time
import os
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

READY_DATA_DIR = r'./data/ready_for_train'
X_train = joblib.load(os.path.join(READY_DATA_DIR, 'X_train.pkl'))
y_train = joblib.load(os.path.join(READY_DATA_DIR, 'y_train.pkl'))

print('Training set size:', X_train.shape)

### 6.1 Custom LightGBM Implementation from Scratch

We build a simplified version of LightGBM focusing on its core optimizations:
1. **Histogram-based Binning:** Pre-discretizes features into $K$ bins to speed up tree split searches. We skip binning for columns with fewer unique values than the bin count (e.g. one-hot encoded features) to avoid losing categorical details.
2. **Leaf-wise Tree Growth:** Grows decision trees node-by-node (best-first) controlled by the `max_leaf_nodes` parameter.
3. **Gradient Boosting Machine:** Sequentially trains decision trees on the negative loss gradients (residuals).

In [ ]:
import sys

class LightGBMFromScratch:
    def __init__(self, n_estimators=50, learning_rate=0.1, max_depth=3, max_leaf_nodes=8, n_bins=32):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.max_leaf_nodes = max_leaf_nodes
        self.n_bins = n_bins
        self.base_pred = None
        self.trees = []
        self.bin_edges = {}

    def _bin_features(self, X, fit=False):
        X_binned = X.copy()
        for col in X.columns:
            if fit:
                n_unique = X[col].nunique()
                if n_unique > self.n_bins:
                    percentiles = np.linspace(0, 100, self.n_bins + 1)
                    edges = np.percentile(X[col], percentiles)
                    edges = np.unique(edges)
                    self.bin_edges[col] = edges
                else:
                    self.bin_edges[col] = None
            edges = self.bin_edges[col]
            if edges is not None:
                X_binned[col] = np.digitize(X[col], edges[1:-1])
        return X_binned

    def fit(self, X, y):
        X_binned = self._bin_features(X, fit=True)
        self.base_pred = np.mean(y)
        f_m = np.full(len(y), self.base_pred)
        
        from sklearn.tree import DecisionTreeRegressor
        self.trees = []
        for i in range(self.n_estimators):
            gradient = y - f_m
            tree = DecisionTreeRegressor(
                max_depth=self.max_depth,
                max_leaf_nodes=self.max_leaf_nodes,
                random_state=42 + i
            )
            tree.fit(X_binned, gradient)
            f_m += self.learning_rate * tree.predict(X_binned)
            self.trees.append(tree)

    def predict(self, X):
        X_binned = self._bin_features(X, fit=False)
        preds = np.full(len(X), self.base_pred)
        for tree in self.trees:
            preds += self.learning_rate * tree.predict(X_binned)
        return preds

# Register in the main system namespace for clean pickling
sys.modules['__main__'].LightGBMFromScratch = LightGBMFromScratch

### 6.2 Initialize Models

In [ ]:
models = {
    'HistGradientBoosting': HistGradientBoostingRegressor(random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'LightGBM': LGBMRegressor(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1),
    'CatBoost': CatBoostRegressor(iterations=100, random_seed=42, verbose=0, allow_writing_files=False),
    'LightGBM (Scratch)': LightGBMFromScratch(n_estimators=50, learning_rate=0.1, max_depth=5, max_leaf_nodes=15, n_bins=32)
}

### 6.3 Train Models

In [ ]:
trained_models = {}
fit_times = {}

for name, model in models.items():
    print(f'Training {name}...')
    start_time = time.time()
    model.fit(X_train, y_train)
    fit_times[name] = time.time() - start_time
    trained_models[name] = model
    print(f'Finished training {name} in {fit_times[name]:.2f} seconds.')

### 6.4 Save Trained Models

In [ ]:
model_dir = r'./data/ready_for_train'

for name, model in trained_models.items():
    clean_name = name.lower().replace(' ', '_').replace('(', '').replace(')', '')
    model_path = os.path.join(model_dir, f'model_{clean_name}.pkl')
    joblib.dump(model, model_path)
    print(f'Saved {name} model to {model_path}')

joblib.dump(fit_times, os.path.join(model_dir, 'fit_times.pkl'))